# Class-29 orbit versus Ineq_cycle

This notebook answers one exact question:

> Among all labeled nonsignaling vertices obtained from the supplied class-29
> representative by the complete relabeling group of the tripartite
> $(2,2;2,2;2,2)$ Bell scenario, which ones violate Ineq_cycle?

Everything is calculated with exact rational arithmetic. No optimization
solver and no nonstandard Julia package is required.

The notebook:

1. reconstructs the class-29 representative from its 26 correlators;
2. encodes Ineq_cycle and its siglocal lower bound 20;
3. enumerates all 3,072 scenario relabelings;
4. applies them to the representative and removes duplicate behaviors;
5. evaluates Ineq_cycle on every distinct orbit member;
6. prints every violating behavior and the relabelings producing it;
7. writes exact TSV files that can be used by the later LP notebooks.

For a clean run in Jupyter, restart the Julia kernel and select Run All.


## 1. Coordinate convention

The 64 probabilities are stored in the order

$$
p(a,b,c\mid x,y,z),\qquad
x,y,z,a,b,c\in\{0,1\},
$$

with loops ordered as

$$
x,\ y,\ z,\ a,\ b,\ c.
$$

Equivalently, the one-based Julia index is

$$
1+c+2b+4a+8z+16y+32x.
$$

Thus the vector consists of eight input blocks, in the order
$000,001,\ldots,111$, and each block contains the eight outputs in the order
$000,001,\ldots,111$.

In the correlator reconstruction, output bit 0 represents the value $-1$ and
output bit 1 represents the value $+1$. This is the convention used in the
collaborator notebook from which the class-29 representative was taken.


In [3]:
# This notebook deliberately uses only Julia Base.

const Rat = Rational{Int}

# All binary triples in lexicographic order.
const TRIPLES = NTuple{3,Int}[
    (u, v, w)
    for u in 0:1
    for v in 0:1
    for w in 0:1
]

# All six permutations of the three parties.
#
# The tuple (2,1,3), for example, means:
#   new Alice receives old Bob,
#   new Bob receives old Alice,
#   new Charlie receives old Charlie.
const PARTY_PERMUTATIONS = NTuple{3,Int}[
    (1, 2, 3),
    (1, 3, 2),
    (2, 1, 3),
    (2, 3, 1),
    (3, 1, 2),
    (3, 2, 1),
]

# Canonical indices used throughout the notebook.
prob_index(a, b, c, x, y, z) =
    1 + c + 2*b + 4*a + 8*z + 16*y + 32*x

abc_index(a, b, c) = 1 + c + 2*b + 4*a
xyz_index(x, y, z) = 1 + z + 2*y + 4*x

# Convert an output bit into the corresponding correlator value.
outcome_sign(a) = a == 0 ? -1 : 1

@assert prob_index(0, 0, 0, 0, 0, 0) == 1
@assert prob_index(1, 1, 1, 1, 1, 1) == 64

println("Coordinate convention initialized.")


Coordinate convention initialized.


## 2. Exact class-29 representative

For a nonsignaling behavior with binary outputs represented by signs
$\alpha,\beta,\gamma\in\{-1,+1\}$,

$$
\begin{aligned}
p(a,b,c\mid x,y,z)=\frac18\bigl(&1+
\alpha A_x+\beta B_y+\gamma C_z\\
&+\alpha\beta\,AB_{xy}
+\alpha\gamma\,AC_{xz}
+\beta\gamma\,BC_{yz}
+\alpha\beta\gamma\,ABC_{xyz}\bigr).
\end{aligned}
$$

The 26 correlators below are the supplied representative of class 29. They
are entered as rational numbers, so the resulting probabilities are exact.


In [5]:
# One-body correlators.
A = Rat[1//3, 1//3]
B = Rat[1//3, 1//3]
C = Rat[1//3, 1//3]

# Two-body correlators. Rows and columns are the corresponding binary inputs.
AB = Rat[
     1//1  -1//3
    -1//3  -1//3
]

AC = Rat[
     1//1  -1//3
    -1//3  -1//3
]

BC = Rat[
     1//1  -1//3
    -1//3  -1//3
]

# Three-body correlators in xyz order:
# 000, 001, 010, 011, 100, 101, 110, 111.
ABC = Rat[
     1//3,  1//3,  1//3, -1//1,
     1//3, -1//1, -1//1, -1//1,
]

function construct_class29_representative(A, B, C, AB, AC, BC, ABC)
    p = fill(0//1, 64)

    for x in 0:1, y in 0:1, z in 0:1
        for a in 0:1, b in 0:1, c in 0:1
            alpha = outcome_sign(a)
            beta  = outcome_sign(b)
            gamma = outcome_sign(c)

            p[prob_index(a, b, c, x, y, z)] =
                (1//8) * (
                    1 +
                    alpha * A[x+1] +
                    beta  * B[y+1] +
                    gamma * C[z+1] +
                    alpha * beta  * AB[x+1, y+1] +
                    alpha * gamma * AC[x+1, z+1] +
                    beta  * gamma * BC[y+1, z+1] +
                    alpha * beta * gamma * ABC[xyz_index(x, y, z)]
                )
        end
    end

    return p
end

p29 = construct_class29_representative(A, B, C, AB, AC, BC, ABC)

println("Class-29 representative reconstructed exactly.")
println("Distinct probability values: ", sort(unique(p29)))


Class-29 representative reconstructed exactly.
Distinct probability values: Rational{Int64}[0, 1//3, 2//3]


### Exact behavior audit

Before using the representative, the next cell checks:

- vector length;
- nonnegativity;
- normalization for every input triple;
- all one-party and two-party nonsignaling conditions.

All equalities are exact: no numerical tolerance is used.


In [7]:
function audit_nonsignaling_behavior(p)
    @assert length(p) == 64
    @assert all(q -> q >= 0//1, p)

    # Normalization for every fixed input triple.
    for x in 0:1, y in 0:1, z in 0:1
        total = sum(
            p[prob_index(a, b, c, x, y, z)]
            for a in 0:1, b in 0:1, c in 0:1
        )
        @assert total == 1//1
    end

    # Alice's marginal must be independent of y and z.
    for x in 0:1, a in 0:1
        reference = sum(
            p[prob_index(a, b, c, x, 0, 0)]
            for b in 0:1, c in 0:1
        )
        for y in 0:1, z in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for b in 0:1, c in 0:1
            )
            @assert current == reference
        end
    end

    # Bob's marginal must be independent of x and z.
    for y in 0:1, b in 0:1
        reference = sum(
            p[prob_index(a, b, c, 0, y, 0)]
            for a in 0:1, c in 0:1
        )
        for x in 0:1, z in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for a in 0:1, c in 0:1
            )
            @assert current == reference
        end
    end

    # Charlie's marginal must be independent of x and y.
    for z in 0:1, c in 0:1
        reference = sum(
            p[prob_index(a, b, c, 0, 0, z)]
            for a in 0:1, b in 0:1
        )
        for x in 0:1, y in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for a in 0:1, b in 0:1
            )
            @assert current == reference
        end
    end

    # AB must be independent of Charlie's input z.
    for x in 0:1, y in 0:1, a in 0:1, b in 0:1
        reference = sum(
            p[prob_index(a, b, c, x, y, 0)]
            for c in 0:1
        )
        for z in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for c in 0:1
            )
            @assert current == reference
        end
    end

    # AC must be independent of Bob's input y.
    for x in 0:1, z in 0:1, a in 0:1, c in 0:1
        reference = sum(
            p[prob_index(a, b, c, x, 0, z)]
            for b in 0:1
        )
        for y in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for b in 0:1
            )
            @assert current == reference
        end
    end

    # BC must be independent of Alice's input x.
    for y in 0:1, z in 0:1, b in 0:1, c in 0:1
        reference = sum(
            p[prob_index(a, b, c, 0, y, z)]
            for a in 0:1
        )
        for x in 0:1
            current = sum(
                p[prob_index(a, b, c, x, y, z)]
                for a in 0:1
            )
            @assert current == reference
        end
    end

    return true
end

@assert audit_nonsignaling_behavior(p29)
println("Exact normalization, positivity, and nonsignaling audit passed.")


Exact normalization, positivity, and nonsignaling audit passed.


## 3. Exact definition of Ineq_cycle

The rows of the displayed matrix follow the input order
$xyz=000,001,\ldots,111$. Its columns follow the output order
$abc=000,001,\ldots,111$.

The valid siglocal inequality is

$$
I_{\mathrm{cycle}}(p)\geq 20.
$$

Consequently, a behavior violates the inequality exactly when its value is
strictly smaller than 20.


In [9]:
Ineq_cycle_matrix = Rat[
    0 12 12 15 12 15 15  0
    0  4 15 12 15 12  5  0
    0 15  4 12 15  5 12  0
    8  4  4  0  0  5  5 10
    0 15 15  5  4 12 12  0
    8  4  0  5  4  0  5 10
    8  0  4  5  4  5  0 10
    5  0  0  4  0  4  4  8
]

function matrix_to_canonical_coefficients(M)
    @assert size(M) == (8, 8)
    coefficients = fill(0//1, 64)

    for x in 0:1, y in 0:1, z in 0:1
        for a in 0:1, b in 0:1, c in 0:1
            coefficients[prob_index(a, b, c, x, y, z)] =
                M[xyz_index(x, y, z), abc_index(a, b, c)]
        end
    end

    return coefficients
end

bell_value(coefficients, p) =
    sum(coefficients[i] * p[i] for i in eachindex(p))

Ineq_cycle = matrix_to_canonical_coefficients(Ineq_cycle_matrix)
Ineq_cycle_bound = 20//1
representative_value = bell_value(Ineq_cycle, p29)

# This fixes the convention before any symmetry is applied.
@assert representative_value == 23//1

println("Ineq_cycle bound: ", Ineq_cycle_bound)
println("Value on the supplied class-29 representative: ", representative_value)
println("The supplied representative itself does not violate the inequality.")


Ineq_cycle bound: 20//1
Value on the supplied class-29 representative: 23//1
The supplied representative itself does not violate the inequality.


## 4. Complete symmetry group

For each party, the local relabelings consist of:

1. swapping its two inputs;
2. independently swapping the two outputs for input 0;
3. independently swapping the two outputs for input 1.

There are therefore $2^3=8$ local relabelings per party. Together with the
six party permutations, the full relabeling group has

$$
6\times 8^3=3072
$$

elements.

The code below enumerates all 3,072 elements directly. For a given party
permutation, the new party $i$ receives the old party indicated by
party_permutation[i]. It then applies the selected input swap and an output
swap conditioned on the new input label.

Each group element is represented as a permutation of the 64 probability
coordinates. The notebook verifies that all 3,072 coordinate permutations
are distinct.


In [11]:
# Decode a mask from 0 to 63 into the six independent output flips:
# (A0, A1, B0, B1, C0, C1).
function output_flips_from_mask(mask)
    return ntuple(
        k -> Int((mask >> (k-1)) & 1),
        6,
    )
end

function coordinate_permutation(
    party_permutation::NTuple{3,Int},
    input_swaps::NTuple{3,Int},
    output_flips::NTuple{6,Int},
)
    permutation = Vector{Int}(undef, 64)

    for x in 0:1, y in 0:1, z in 0:1
        for a in 0:1, b in 0:1, c in 0:1
            old_inputs  = (x, y, z)
            old_outputs = (a, b, c)

            # First permute the parties, then relabel each new party's input.
            new_inputs = ntuple(
                i -> old_inputs[party_permutation[i]] ⊻ input_swaps[i],
                3,
            )

            # Output flips are indexed by the new party and its new input.
            new_outputs = ntuple(
                i -> begin
                    flip_index = 2*(i-1) + new_inputs[i] + 1
                    old_outputs[party_permutation[i]] ⊻ output_flips[flip_index]
                end,
                3,
            )

            old_index = prob_index(
                old_outputs[1], old_outputs[2], old_outputs[3],
                old_inputs[1], old_inputs[2], old_inputs[3],
            )

            new_index = prob_index(
                new_outputs[1], new_outputs[2], new_outputs[3],
                new_inputs[1], new_inputs[2], new_inputs[3],
            )

            permutation[old_index] = new_index
        end
    end

    # Every scenario relabeling must be a bijection of the 64 coordinates.
    @assert sort(permutation) == collect(1:64)
    return Tuple(permutation)
end

symmetry_permutations = NTuple{64,Int}[]
symmetry_descriptions = NamedTuple[]
seen_symmetries = Set{NTuple{64,Int}}()

for party_permutation in PARTY_PERMUTATIONS
    for input_swaps in TRIPLES
        for mask in 0:63
            output_flips = output_flips_from_mask(mask)
            permutation = coordinate_permutation(
                party_permutation,
                input_swaps,
                output_flips,
            )

            # The direct parameterization should produce every group element
            # exactly once.
            @assert !(permutation in seen_symmetries)
            push!(seen_symmetries, permutation)
            push!(symmetry_permutations, permutation)
            push!(
                symmetry_descriptions,
                (
                    party_permutation = party_permutation,
                    input_swaps = input_swaps,
                    output_flips = output_flips,
                ),
            )
        end
    end
end

@assert length(symmetry_permutations) == 3072
@assert length(seen_symmetries) == 3072

println("Complete symmetry group generated: 3072 distinct elements.")


Complete symmetry group generated: 3072 distinct elements.


## 5. Generate and deduplicate the class-29 orbit

Applying all 3,072 group elements does not generally produce 3,072 distinct
behaviors, because the representative can be invariant under some
relabelings. Such relabelings form its stabilizer.

The next cell applies every group element, uses the exact 64-component
rational vector as the dictionary key, and therefore removes duplicates
without rounding.


In [13]:
function apply_coordinate_permutation(
    p::Vector{Rat},
    permutation::NTuple{64,Int},
)
    q = Vector{Rat}(undef, 64)

    # permutation[old_index] is the coordinate occupied by that probability
    # after the relabeling.
    for old_index in 1:64
        q[permutation[old_index]] = p[old_index]
    end

    return q
end

orbit = Vector{Vector{Rat}}()
orbit_lookup = Dict{NTuple{64,Rat},Int}()
first_symmetry_for_orbit_member = NamedTuple[]
symmetry_multiplicity = Int[]

# For each of the 3,072 group elements, record the corresponding distinct
# orbit-member identifier. This later lets us recover every transformation
# that produces a violating behavior.
symmetry_orbit_ids = Vector{Int}(undef, length(symmetry_permutations))

for group_id in eachindex(symmetry_permutations)
    q = apply_coordinate_permutation(
        p29,
        symmetry_permutations[group_id],
    )
    key = Tuple(q)
    orbit_id = get(orbit_lookup, key, 0)

    if orbit_id == 0
        push!(orbit, q)
        orbit_id = length(orbit)
        orbit_lookup[key] = orbit_id
        push!(
            first_symmetry_for_orbit_member,
            symmetry_descriptions[group_id],
        )
        push!(symmetry_multiplicity, 1)
    else
        symmetry_multiplicity[orbit_id] += 1
    end

    symmetry_orbit_ids[group_id] = orbit_id
end

# Exact internal checks.
@assert length(orbit) == 512
@assert sum(symmetry_multiplicity) == 3072
@assert all(m -> m == 6, symmetry_multiplicity)

# Scenario relabelings preserve normalization and nonsignaling.
# We nevertheless audit every distinct orbit member explicitly.
for q in orbit
    @assert audit_nonsignaling_behavior(q)
end

println("Number of distinct class-29 orbit members: ", length(orbit))
println("Stabilizer size: ", 3072 ÷ length(orbit))
println("Every orbit member passed the exact behavior audit.")


Number of distinct class-29 orbit members: 512
Stabilizer size: 6
Every orbit member passed the exact behavior audit.


## 6. Evaluate Ineq_cycle on every distinct orbit member

The value and violation are computed exactly:

$$
\text{signed violation}=20-I_{\mathrm{cycle}}(p).
$$

A positive signed violation means that the lower-bound inequality is
violated.


In [15]:
orbit_values = Rat[
    bell_value(Ineq_cycle, q)
    for q in orbit
]

signed_violations = Rat[
    Ineq_cycle_bound - value
    for value in orbit_values
]

violating_orbit_ids = findall(
    value -> value < Ineq_cycle_bound,
    orbit_values,
)

saturating_orbit_ids = findall(
    value -> value == Ineq_cycle_bound,
    orbit_values,
)

# These regression checks are based on an independent exact calculation.
@assert minimum(orbit_values) == 17//1
@assert maximum(orbit_values) == 69//1
@assert length(violating_orbit_ids) == 1
@assert isempty(saturating_orbit_ids)

println("Minimum orbit value: ", minimum(orbit_values))
println("Maximum orbit value: ", maximum(orbit_values))
println("Number of violating orbit members: ", length(violating_orbit_ids))
println("Number of saturating orbit members: ", length(saturating_orbit_ids))
println("Violating orbit identifiers: ", violating_orbit_ids)

# Exact value histogram. This is useful for spotting accidental changes in
# conventions or in the group action.
value_histogram = Dict{Rat,Int}()
for value in orbit_values
    value_histogram[value] = get(value_histogram, value, 0) + 1
end

println()
println("Exact Ineq_cycle value histogram:")
for value in sort(collect(keys(value_histogram)))
    println("  ", value, "  ->  ", value_histogram[value], " behavior(s)")
end


Minimum orbit value: 17//1
Maximum orbit value: 69//1
Number of violating orbit members: 1
Number of saturating orbit members: 0
Violating orbit identifiers: [64]

Exact Ineq_cycle value histogram:
  17//1  ->  1 behavior(s)
  23//1  ->  11 behavior(s)
  26//1  ->  4 behavior(s)
  89//3  ->  3 behavior(s)
  32//1  ->  6 behavior(s)
  98//3  ->  6 behavior(s)
  103//3  ->  6 behavior(s)
  35//1  ->  6 behavior(s)
  112//3  ->  3 behavior(s)
  38//1  ->  12 behavior(s)
  116//3  ->  3 behavior(s)
  121//3  ->  3 behavior(s)
  127//3  ->  9 behavior(s)
  44//1  ->  18 behavior(s)
  134//3  ->  24 behavior(s)
  136//3  ->  6 behavior(s)
  137//3  ->  12 behavior(s)
  139//3  ->  36 behavior(s)
  47//1  ->  36 behavior(s)
  143//3  ->  18 behavior(s)
  145//3  ->  6 behavior(s)
  146//3  ->  6 behavior(s)
  148//3  ->  6 behavior(s)
  50//1  ->  36 behavior(s)
  152//3  ->  6 behavior(s)
  154//3  ->  3 behavior(s)
  155//3  ->  3 behavior(s)
  157//3  ->  6 behavior(s)
  53//1  ->  6 behav

## 7. Pinpoint every violating behavior

For each violator, the next cell prints:

- its orbit identifier;
- its exact Bell value and signed violation;
- a complete $8\times8$ probability matrix;
- every symmetry transformation that maps the supplied representative to it.

The matrix rows are labeled by $xyz$ and its columns by $abc$, both in
lexicographic order.


In [17]:
function behavior_as_matrix(p)
    M = Matrix{Rat}(undef, 8, 8)

    for x in 0:1, y in 0:1, z in 0:1
        for a in 0:1, b in 0:1, c in 0:1
            M[xyz_index(x, y, z), abc_index(a, b, c)] =
                p[prob_index(a, b, c, x, y, z)]
        end
    end

    return M
end

for orbit_id in violating_orbit_ids
    println()
    println("============================================================")
    println("Violating orbit member ", orbit_id)
    println("Ineq_cycle value: ", orbit_values[orbit_id])
    println("Signed violation 20 - value: ", signed_violations[orbit_id])
    println()
    println("Probability matrix: rows xyz, columns abc")
    display(behavior_as_matrix(orbit[orbit_id]))

    producing_group_ids = findall(
        id -> id == orbit_id,
        symmetry_orbit_ids,
    )

    println()
    println(
        "Number of group elements producing this behavior: ",
        length(producing_group_ids),
    )

    for group_id in producing_group_ids
        description = symmetry_descriptions[group_id]
        println(
            "  group_id=", group_id,
            ", party_permutation=", description.party_permutation,
            ", input_swaps=", description.input_swaps,
            ", output_flips(A0,A1,B0,B1,C0,C1)=",
            description.output_flips,
        )
    end
end



Violating orbit member 64
Ineq_cycle value: 17//1
Signed violation 20 - value: 3//1

Probability matrix: rows xyz, columns abc


8×8 Matrix{Rational{Int64}}:
 2//3   0     0     0     0     0     0    1//3
 1//3  1//3   0     0     0     0    1//3   0
 1//3   0    1//3   0     0    1//3   0     0
  0    1//3  1//3   0    1//3   0     0     0
 1//3   0     0    1//3  1//3   0     0     0
  0    1//3  1//3   0    1//3   0     0     0
  0    1//3  1//3   0    1//3   0     0     0
  0    1//3  1//3   0    1//3   0     0     0


Number of group elements producing this behavior: 6
  group_id=64, party_permutation=(1, 2, 3), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)
  group_id=576, party_permutation=(1, 3, 2), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)
  group_id=1088, party_permutation=(2, 1, 3), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)
  group_id=1600, party_permutation=(2, 3, 1), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)
  group_id=2112, party_permutation=(3, 1, 2), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)
  group_id=2624, party_permutation=(3, 2, 1), input_swaps=(0, 0, 0), output_flips(A0,A1,B0,B1,C0,C1)=(1, 1, 1, 1, 1, 1)


## 8. Save exact results

Four TSV files are written to the folder
class29_orbit_Ineq_cycle_results:

1. class29_orbit_values.tsv: one row for each of the 512 distinct behaviors;
2. class29_violating_behaviors_wide.tsv: one complete 64-coordinate vector
   per violator, suitable for the later LP stage;
3. class29_violating_behaviors_long.tsv: the same probabilities with explicit
   $a,b,c,x,y,z$ labels;
4. class29_violating_transformations.tsv: every relabeling that produces each
   violator.

Exact rational values are stored as strings such as 1/3. Floating-point
columns are included only for convenience.


In [19]:
const OUTPUT_DIRECTORY = "class29_orbit_Ineq_cycle_results"
mkpath(OUTPUT_DIRECTORY)

function rational_text(q::Rat)
    if denominator(q) == 1
        return string(numerator(q))
    end
    return string(numerator(q), "/", denominator(q))
end

tuple_text(t) = join(string.(collect(t)), ",")

# ------------------------------------------------------------
# 1. Summary of all 512 distinct orbit members.
# ------------------------------------------------------------
orbit_values_path = joinpath(
    OUTPUT_DIRECTORY,
    "class29_orbit_values.tsv",
)

open(orbit_values_path, "w") do io
    println(
        io,
        join(
            [
                "orbit_id",
                "Ineq_cycle_exact",
                "Ineq_cycle_float",
                "signed_violation_exact",
                "signed_violation_float",
                "violates",
                "symmetry_multiplicity",
                "first_party_permutation",
                "first_input_swaps",
                "first_output_flips_A0_A1_B0_B1_C0_C1",
            ],
            "\t",
        ),
    )

    for orbit_id in eachindex(orbit)
        description = first_symmetry_for_orbit_member[orbit_id]
        fields = String[
            string(orbit_id),
            rational_text(orbit_values[orbit_id]),
            string(Float64(orbit_values[orbit_id])),
            rational_text(signed_violations[orbit_id]),
            string(Float64(signed_violations[orbit_id])),
            string(orbit_values[orbit_id] < Ineq_cycle_bound),
            string(symmetry_multiplicity[orbit_id]),
            tuple_text(description.party_permutation),
            tuple_text(description.input_swaps),
            tuple_text(description.output_flips),
        ]
        println(io, join(fields, "\t"))
    end
end

# Canonical coordinate names for the wide table.
coordinate_names = String[]
for x in 0:1, y in 0:1, z in 0:1
    for a in 0:1, b in 0:1, c in 0:1
        push!(
            coordinate_names,
            "p$(a)$(b)$(c)_given_$(x)$(y)$(z)",
        )
    end
end
@assert length(coordinate_names) == 64

# ------------------------------------------------------------
# 2. One wide 64-coordinate row for each violator.
# ------------------------------------------------------------
wide_path = joinpath(
    OUTPUT_DIRECTORY,
    "class29_violating_behaviors_wide.tsv",
)

open(wide_path, "w") do io
    header = vcat(
        ["orbit_id", "Ineq_cycle_exact", "signed_violation_exact"],
        coordinate_names,
    )
    println(io, join(header, "\t"))

    for orbit_id in violating_orbit_ids
        fields = vcat(
            String[
                string(orbit_id),
                rational_text(orbit_values[orbit_id]),
                rational_text(signed_violations[orbit_id]),
            ],
            rational_text.(orbit[orbit_id]),
        )
        println(io, join(fields, "\t"))
    end
end

# ------------------------------------------------------------
# 3. Coordinate-labeled long representation of every violator.
# ------------------------------------------------------------
long_path = joinpath(
    OUTPUT_DIRECTORY,
    "class29_violating_behaviors_long.tsv",
)

open(long_path, "w") do io
    println(
        io,
        join(
            [
                "orbit_id", "coordinate_index",
                "a", "b", "c", "x", "y", "z",
                "probability_exact", "numerator", "denominator",
                "probability_float",
            ],
            "\t",
        ),
    )

    for orbit_id in violating_orbit_ids
        p = orbit[orbit_id]

        for x in 0:1, y in 0:1, z in 0:1
            for a in 0:1, b in 0:1, c in 0:1
                index = prob_index(a, b, c, x, y, z)
                q = p[index]
                fields = String[
                    string(orbit_id),
                    string(index),
                    string(a), string(b), string(c),
                    string(x), string(y), string(z),
                    rational_text(q),
                    string(numerator(q)),
                    string(denominator(q)),
                    string(Float64(q)),
                ]
                println(io, join(fields, "\t"))
            end
        end
    end
end

# ------------------------------------------------------------
# 4. Every group element that produces a violating behavior.
# ------------------------------------------------------------
transformations_path = joinpath(
    OUTPUT_DIRECTORY,
    "class29_violating_transformations.tsv",
)

open(transformations_path, "w") do io
    println(
        io,
        join(
            [
                "orbit_id", "group_id",
                "party_permutation",
                "input_swaps",
                "output_flips_A0_A1_B0_B1_C0_C1",
            ],
            "\t",
        ),
    )

    violating_set = Set(violating_orbit_ids)
    for group_id in eachindex(symmetry_descriptions)
        orbit_id = symmetry_orbit_ids[group_id]

        if orbit_id in violating_set
            description = symmetry_descriptions[group_id]
            fields = String[
                string(orbit_id),
                string(group_id),
                tuple_text(description.party_permutation),
                tuple_text(description.input_swaps),
                tuple_text(description.output_flips),
            ]
            println(io, join(fields, "\t"))
        end
    end
end

println("Saved:")
println("  ", orbit_values_path)
println("  ", wide_path)
println("  ", long_path)
println("  ", transformations_path)


Saved:
  class29_orbit_Ineq_cycle_results\class29_orbit_values.tsv
  class29_orbit_Ineq_cycle_results\class29_violating_behaviors_wide.tsv
  class29_orbit_Ineq_cycle_results\class29_violating_behaviors_long.tsv
  class29_orbit_Ineq_cycle_results\class29_violating_transformations.tsv


## Expected conclusion

With the conventions fixed above, a clean run should report:

- 3,072 distinct symmetry transformations;
- 512 distinct behaviors in the class-29 orbit;
- stabilizer size 6;
- value 23 for the originally supplied representative;
- exactly one distinct orbit member violating Ineq_cycle;
- minimum orbit value 17, hence an exact violation of magnitude 3.

The assertions deliberately stop execution if any of these independently
checked results changes. This protects the later LP stage from silently using
an inconsistent coordinate order or symmetry action.

The existence of this violating class-29 orbit member makes it possible for a
separation LP to return Ineq_cycle, but it does not guarantee that a particular
LP normalization or objective will select that inequality. That question is
reserved for the next stage.
